# Data Quality & Governed KPIs

Demonstrates quality gates before business metrics are published.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
fact = pd.read_csv(ROOT/'data/raw/hourly_operations.csv', parse_dates=['timestamp'])
assets = pd.read_csv(ROOT/'data/raw/assets.csv')


In [2]:
checks = {
 'unique_asset_hour': ~fact.duplicated(['timestamp','asset_id']).any(),
 'required_keys_complete': fact[['timestamp','asset_id']].notna().all().all(),
 'generation_nonnegative': (fact.generation_mwh >= 0).all(),
 'generation_within_nameplate': (fact.generation_mwh <= fact.capacity_mw + 1e-8).all(),
 'availability_binary': fact.available_flag.isin([0,1]).all()
}
pd.Series(checks, name='passed')

unique_asset_hour              True
required_keys_complete         True
generation_nonnegative         True
generation_within_nameplate    True
availability_binary            True
Name: passed, dtype: bool

## Canonical monthly portfolio KPIs

In [3]:
monthly = fact.assign(month=fact.timestamp.dt.to_period('M').astype(str)).groupby('month', as_index=False).agg(generation_mwh=('generation_mwh','sum'), revenue_usd=('market_revenue_usd','sum'), gross_margin_usd=('gross_margin_usd','sum'), curtailment_mwh=('curtailment_mwh','sum'))
monthly['margin_per_mwh'] = monthly.gross_margin_usd/monthly.generation_mwh
monthly

,month,generation_mwh,revenue_usd,gross_margin_usd,curtailment_mwh,margin_per_mwh
0,2026-01,758616.382462,2.929820e+07,1.217767e+07,1.877233,16.052470
1,2026-02,661643.619448,2.464666e+07,1.008391e+07,0.000000,15.240698
2,2026-03,732302.316960,2.688007e+07,1.136101e+07,4.062795,15.514102
3,2026-04,716542.264899,2.590760e+07,1.150958e+07,11.862660,16.062671
4,2026-05,771728.765264,2.636134e+07,1.146393e+07,17.142510,14.854867
5,2026-06,755003.961059,2.625939e+07,1.220198e+07,11.462150,16.161475


## Governance principle

A KPI should have one business definition, one canonical calculation, and clear lineage from the hourly fact table to the executive scorecard. The same definitions are mirrored in `docs/KPI_DICTIONARY.md`, SQL, and DAX.